# Notebook 05 — Shoreline Transects and Coastal-Change Preparation


This notebook creates shore-normal transects for the 2024 shoreline.

Transects will later be used to measure shoreline position and change.

This notebook does not calculate change yet because only 2024 is processed.

The unresolved coastal NoData QA observation remains open.


In [1]:
from pathlib import Path

import geopandas as gpd
import folium
import numpy as np
import pandas as pd
from shapely.geometry import LineString, MultiLineString
from shapely.ops import linemerge

# --- Project root (robust to where the notebook is launched from) ---
cwd = Path.cwd()
if (cwd / "config").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "config").exists() and (cwd.parent / "notebooks").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve PROJECT_ROOT (expected config/ and notebooks/ alongside).")

# --- Configuration ---
DATA_DIR = PROJECT_ROOT / "data"
STUDY_DIR = DATA_DIR / "processed" / "mombasa_test"
YEAR_DIR = STUDY_DIR / "2024"
SHORELINE_PATH = YEAR_DIR / "shoreline_2024_shared_boundary_test.gpkg"
STUDY_AREA_PATH = PROJECT_ROOT / "config" / "study_area.geojson"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transects" / "2024"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHORELINE_LAYER = "shoreline_candidates"

# Metric CRS for the Mombasa area (Kenya, UTM zone 37S)
UTM_CRS = "EPSG:32737"
WGS84 = "EPSG:4326"

# Transect parameters
TRANSECT_SPACING_M = 500.0
TRANSECT_OFFSET_M = 1000.0
SIMPLIFY_TOLERANCE_M = 10.0  # conservative simplification tolerance

print(f"Project root: {PROJECT_ROOT}")
print(f"Shoreline: {SHORELINE_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


Project root: c:\Users\pc\Documents\Data Science Projects\Shoremonitor
Shoreline: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\processed\mombasa_test\2024\shoreline_2024_shared_boundary_test.gpkg
Output dir: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024


In [2]:
print("--- Available layers in shoreline GeoPackage ---")
try:
    layers_info = gpd.list_layers(SHORELINE_PATH)
    if hasattr(layers_info, "__getitem__") and "name" in layers_info.columns:
        layer_names = layers_info["name"].tolist()
    else:
        layer_names = [row.name for row in layers_info.itertuples()]
except Exception as exc:  # pragma: no cover - fallback when layer listing is unsupported
    print(f"(Could not list layers: {exc})")
    layer_names = [SHORELINE_LAYER]
for name in layer_names:
    print(f"- {name}")

if not SHORELINE_PATH.exists():
    raise SystemExit(f"ERROR: required input not found: {SHORELINE_PATH}")

gdf = gpd.read_file(SHORELINE_PATH, layer=SHORELINE_LAYER)

print(f"\nShoreline CRS: {gdf.crs}")
print(f"Shoreline feature count: {len(gdf)}")
print(f"Shoreline bounds: {gdf.total_bounds}")
print(f"Shoreline geometry types: {gdf.geometry.type.unique().tolist()}")

if not STUDY_AREA_PATH.exists():
    raise SystemExit(f"ERROR: required input not found: {STUDY_AREA_PATH}")

study_area = gpd.read_file(STUDY_AREA_PATH)
print(f"\nStudy area CRS: {study_area.crs}")
print(f"Study area feature count: {len(study_area)}")
print(f"Study area bounds: {study_area.total_bounds}")
print(f"Study area geometry types: {study_area.geometry.type.unique().tolist()}")

if gdf.empty:
    raise SystemExit("ERROR: shoreline_candidates layer is empty. Cannot create transects.")


--- Available layers in shoreline GeoPackage ---
- shoreline_candidates

Shoreline CRS: EPSG:32737
Shoreline feature count: 15802
Shoreline bounds: [ 572150. 9546790.  594340. 9568830.]
Shoreline geometry types: ['LineString']

Study area CRS: EPSG:4326
Study area feature count: 1
Study area bounds: [39.65 -4.1  39.85 -3.9 ]
Study area geometry types: ['Polygon']


In [3]:
print("--- Creating coastal baseline ---")

# Work in a metric CRS for transect geometry
shoreline_utm = gdf.to_crs(UTM_CRS)

# Dissolve all shoreline candidates into a single geometry
dissolved = shoreline_utm.geometry.unary_union

# Merge into a single line where possible
baseline = linemerge(dissolved)
if isinstance(baseline, MultiLineString):
    parts = [p for p in baseline.geoms if p.length > 0]
    if not parts:
        raise SystemExit("ERROR: dissolved shoreline produced no usable line geometry.")
    baseline = max(parts, key=lambda g: g.length)

# Conservative smoothing + simplification (does not alter the source candidates)
baseline = baseline.simplify(tolerance=SIMPLIFY_TOLERANCE_M, preserve_topology=True)
if isinstance(baseline, MultiLineString):
    parts = [p for p in baseline.geoms if p.length > 0]
    baseline = max(parts, key=lambda g: g.length)
if baseline.geom_type != "LineString":
    raise SystemExit(f"ERROR: baseline is not a LineString: {baseline.geom_type}")

baseline_utm = gpd.GeoDataFrame(geometry=[baseline], crs=UTM_CRS)
baseline_path = OUTPUT_DIR / "shoreline_baseline_2024.gpkg"
baseline_utm.to_file(baseline_path, driver="GPKG", layer="shoreline_baseline")

print(f"Baseline geometry type: {baseline.geom_type}")
print(f"Baseline length: {baseline.length:.2f} m")
print(f"Baseline bounds: {baseline.bounds}")
print(f"Saved baseline: {baseline_path}")


--- Creating coastal baseline ---


C:\Users\pc\AppData\Local\Temp\ipykernel_316\2780625948.py:7: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  dissolved = shoreline_utm.geometry.unary_union


Baseline geometry type: LineString
Baseline length: 10801.33 m
Baseline bounds: (572160.0, 9553120.0, 575260.0, 9555240.0)
Saved baseline: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024\shoreline_baseline_2024.gpkg


In [4]:
print("--- Generating shore-normal transects ---")

total_length = baseline.length
spacing = TRANSECT_SPACING_M
offset = TRANSECT_OFFSET_M

distances = list(range(0, int(total_length), int(spacing)))
if int(total_length) not in distances:
    distances.append(int(total_length))

transects = []
transect_ids = []
for i, d in enumerate(distances):
    d = float(d)
    pt = baseline.interpolate(d)

    # Shore-normal direction from the local baseline tangent
    eps = max(1.0, 0.01 * spacing)
    d0 = max(0.0, d - eps)
    d1 = min(total_length, d + eps)
    p0 = baseline.interpolate(d0)
    p1 = baseline.interpolate(d1)
    dx = p1.x - p0.x
    dy = p1.y - p0.y
    seg_len = (dx * dx + dy * dy) ** 0.5
    if seg_len == 0:
        continue
    nx, ny = -dy / seg_len, dx / seg_len  # unit normal

    x, y = pt.x, pt.y
    line = LineString([
        (x - offset * nx, y - offset * ny),
        (x + offset * nx, y + offset * ny),
    ])
    transects.append(line)
    transect_ids.append(f"T{i:04d}")

transect_gdf = gpd.GeoDataFrame(
    {"transect_id": transect_ids},
    geometry=transects,
    crs=UTM_CRS,
)

transect_path = OUTPUT_DIR / "shoreline_transects_2024.gpkg"
transect_gdf.to_file(transect_path, driver="GPKG", layer="transects")

print(f"Transects created: {len(transect_gdf)}")
print(f"Saved transects: {transect_path}")


--- Generating shore-normal transects ---
Transects created: 23
Saved transects: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024\shoreline_transects_2024.gpkg


In [5]:
print("--- Building inspection map ---")

baseline_wgs = baseline_utm.to_crs(WGS84)
transects_wgs = transect_gdf.to_crs(WGS84)
shoreline_wgs = gdf.to_crs(WGS84)
study_wgs = study_area.to_crs(WGS84)

# Map center from study area bounds
minx, miny, maxx, maxy = study_wgs.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, zoom_start=13, control_scale=True)

folium.GeoJson(
    baseline_wgs,
    name="Baseline (2024)",
    style_function=lambda x: {"color": "blue", "weight": 3, "opacity": 0.9},
).add_to(m)

folium.GeoJson(
    transects_wgs,
    name="Transects (2024)",
    style_function=lambda x: {"color": "red", "weight": 1, "opacity": 0.6},
).add_to(m)

folium.GeoJson(
    shoreline_wgs,
    name="Shoreline candidates (2024)",
    style_function=lambda x: {"color": "green", "weight": 2, "opacity": 0.7},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_path = OUTPUT_DIR / "shoreline_transects_map.html"
m.save(str(map_path))

print(f"Map saved: {map_path}")


--- Building inspection map ---
Map saved: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024\shoreline_transects_map.html


In [6]:
print("--- Transect summary table ---")

lengths_m = transect_gdf.geometry.length  # metres (UTM)
centroids = transects_wgs.geometry.centroid  # lon/lat (WGS84)

summary = pd.DataFrame({
    "transect_id": transect_gdf["transect_id"],
    "length_m": lengths_m.values,
    "centroid_lon": centroids.x.values,
    "centroid_lat": centroids.y.values,
})

csv_path = OUTPUT_DIR / "shoreline_transects_2024.csv"
summary.to_csv(csv_path, index=False)

print(f"Summary rows: {len(summary)}")
print(f"Saved CSV: {csv_path}")

print("\nFirst 10 rows:")
from IPython.display import display
display(summary.head(10))


--- Transect summary table ---
Summary rows: 23
Saved CSV: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\transects\2024\shoreline_transects_2024.csv

First 10 rows:


C:\Users\pc\AppData\Local\Temp\ipykernel_316\2743415009.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = transects_wgs.geometry.centroid  # lon/lat (WGS84)


,transect_id,length_m,centroid_lon,centroid_lat
0,T0000,2000.0,39.672943,-4.024172
1,T0001,2000.0,39.671013,-4.024083
2,T0002,2000.0,39.671232,-4.026974
3,T0003,2000.0,39.672038,-4.030268
4,T0004,2000.0,39.673310,-4.032026
5,T0005,2000.0,39.676625,-4.031643
6,T0006,2000.0,39.677636,-4.035340
7,T0007,2000.0,39.677459,-4.038979
8,T0008,2000.0,39.674939,-4.041179
9,T0009,2000.0,39.672528,-4.042717


Transects are prepared for future shoreline-position and change analysis.

No change rates (erosion, accretion, retreat) are calculated in this notebook:
shoreline positions are only available for 2024, so no multi-year
change analysis is possible yet.

The coastal NoData QA observation remains unresolved and should be
addressed before relying on these transects for change measurement.
